In [2]:
# ============================================================================
# notebook: notebooks/04_group_tests.ipynb  (v12 — A' two diagnostics)
# Project: "Incidental vs. Engineered Approval"
# Stage 4 (A'): cross-group tests. There is NO composite EngineeredScore.
#   The group question lives on DIAGNOSTIC 1 (Stability): do disadvantaged and
#   advantaged borderline approvals differ in local stability?
#   Diagnostic 2 (density/paradox) is a within-cohort risk signal, not a group
#   contrast, so it is reported in Stage 5 / 07, not here.
#   Tests:
#     T1. dis_primary vs advantaged on Stability (main) + appendix metrics.
#     T2. DROPPED (R-17): fixed-smaller-group subsampling understated variance.
#     T3. RQ3 — bounded claim: a stability gap approval-rate metrics miss.
#     T4. per-cell robustness (3 primary disadvantaged cells vs advantaged).
# Reads results/. Run from notebooks/.
# ============================================================================


# ---------------------------------------------------------------------------
# CELL 1 — Paths, imports, load Stage-3 per-diagnostic borderline set
# ---------------------------------------------------------------------------
import warnings; warnings.filterwarnings("ignore")
from pathlib import Path
import numpy as np
import pandas as pd
from scipy import stats

ROOT    = Path("..").resolve()
RESULTS = ROOT / "results"
RANDOM_STATE = 42
rng = np.random.default_rng(RANDOM_STATE)

B = pd.read_parquet(RESULTS / "stage3_borderline_scored.parquet")
# Diagnostic 1 is the group-contrast axis. The rest are appendix / reference.
MAIN     = ["A_Stability"]                          # Diagnostic 1 (the group test)
APPENDIX = ["density_pct", "A_LowDensity", "A_NonFrag"]
print(f"Borderline set: {len(B)}")
print("Groups:", B["GROUP"].value_counts().to_dict())


# ---------------------------------------------------------------------------
# CELL 2 — T1: dis_primary vs advantaged. Main = Stability; appendix = rest.
# Mann-Whitney U (non-parametric) + rank-biserial effect size.
# ---------------------------------------------------------------------------
dis = B[B["GROUP"] == "dis_primary"]
adv = B[B["GROUP"] == "advantaged"]
print(f"T1 — dis_primary (n={len(dis)}) vs advantaged (n={len(adv)})\n")

def rank_biserial(u, n1, n2):
    return 1 - (2*u) / (n1*n2)   # effect size for Mann-Whitney

def run(metrics, tag):
    rows = []
    for m in metrics:
        x, y = dis[m].values, adv[m].values
        u, p = stats.mannwhitneyu(x, y, alternative="two-sided")
        rb = rank_biserial(u, len(x), len(y))
        rows.append((m, x.mean(), y.mean(), x.mean()-y.mean(), rb, p))
        star = " *" if p < 0.05 else ""
        print(f"  [{tag}] {m:16} dis={x.mean():.3f} adv={y.mean():.3f} "
              f"diff={x.mean()-y.mean():+.3f}  rb={rb:+.3f}  p={p:.3e}{star}")
    return rows

print(">> MAIN — Diagnostic 1 (Stability)")
t1_main = run(MAIN, "MAIN")
print("\n>> APPENDIX (density is a risk signal not a group contrast; "
      "LowDensity/NonFrag not independent axes)")
t1_app = run(APPENDIX, "APPX")
t1 = pd.DataFrame(t1_main + t1_app,
                  columns=["metric","dis","adv","diff","rank_biserial","p"])


# ---------------------------------------------------------------------------
# CELL 3 — T2 DROPPED (R-17).
# The old subsampling control downsampled only the larger group and held the
# smaller group fixed, understating variability. Gap CIs come from the two-sided
# replacement bootstrap in 07_robustness.ipynb.
# ---------------------------------------------------------------------------
print("T2 subsampling control is DROPPED (R-17).")
print("Reason: fixing the smaller group understates variability; the two-sided")
print("bootstrap in 07_robustness.ipynb is used for the Stability gap CI.")


# ---------------------------------------------------------------------------
# CELL 4 — T3 (RQ3): bounded claim. Groups were defined by low approval rate AND
# demographic cells, so we claim only that a stability gap exists which
# approval-rate metrics do not surface — NOT a fully independent layer.
# ---------------------------------------------------------------------------
print("T3 (RQ3) — stability gap vs the approval-rate gap (BOUNDED):")
print("  Groups were defined by low approval rate AND demographic cells.")
print("  Bounded claim: a stability difference exists that approval-rate")
print("  metrics do not surface — NOT a fully independent layer.\n")
m = "A_Stability"
diff = dis[m].mean() - adv[m].mean()
_, p = stats.mannwhitneyu(dis[m], adv[m], alternative="two-sided")
print(f"  {m:16} gap={diff:+.3f} (p={p:.3e})  "
      f"{'stability signal present' if p<0.05 else 'no separate signal'}")


# ---------------------------------------------------------------------------
# CELL 5 — T4: per-cell robustness (3 primary disadvantaged cells vs advantaged).
# Main focus: Stability. Appendix metrics shown but flagged.
# ---------------------------------------------------------------------------
PRIMARY = ["M·20s·univ", "F·20s·univ", "M·30s·univ"]
SHOW = MAIN + APPENDIX
print("T4 — per-cell: each primary disadvantaged cell vs advantaged pool")
print(f"{'cell':>14} {'n':>4} " + " ".join(f"{s[:9]:>10}" for s in SHOW))
for cell in PRIMARY:
    c = B[B["CELL"] == cell]
    if len(c) < 20:
        print(f"  {cell:>14} {len(c):>4}  (too few)"); continue
    row = f"  {cell:>14} {len(c):>4} "
    for s in SHOW:
        _, p = stats.mannwhitneyu(c[s], adv[s], alternative="two-sided")
        d = c[s].mean() - adv[s].mean()
        row += f" {d:+.2f}{'*' if p<0.05 else ' '}"
    print(row)
print("  (first column = Stability [main]; the rest are appendix)")


# ---------------------------------------------------------------------------
# CELL 6 — Stage 4 verdict (Diagnostic 1: Stability group contrast)
# ---------------------------------------------------------------------------
st = t1[t1.metric == "A_Stability"].iloc[0]
print("=" * 70)
print("STAGE 4 — RQ2/RQ3 VERDICT (v12, A' — Diagnostic 1)")
print("=" * 70)
print(f"Stability gap (dis - adv): {st['diff']:+.3f} (p={st['p']:.3e}) "
      f"-> {'significant (more stable)' if st['p']<0.05 else 'not significant'}")
print("-" * 70)
print("FINDING (Diagnostic 1): disadvantaged borderline approvals are more")
print("  locally STABLE than advantaged ones. This is a group difference in the")
print("  stability profile, not in an overall reliability amount (there is no")
print("  valid composite). Gap CI confirmed by the two-sided bootstrap (07).")
print("  Note: Stability's relation to default is documented in Stage 3 (C7) and")
print("  is a separate matter from this group contrast.")
print("=" * 70)

t1.to_csv(RESULTS / "stage4_t1_group_tests_v12.csv", index=False)
print("Saved -> results/stage4_t1_group_tests_v12.csv  (no T2, no ensemble)")

Borderline set: 1141
Groups: {'other': 547, 'dis_primary': 254, 'advantaged': 224, 'dis_secondary': 112, 'highlight_F60': 4}
T1 — dis_primary (n=254) vs advantaged (n=224)

>> MAIN — Diagnostic 1 (Stability)
  [MAIN] A_Stability      dis=0.328 adv=0.242 diff=+0.086  rb=-0.229  p=1.602e-05 *

>> APPENDIX (density is a risk signal not a group contrast; LowDensity/NonFrag not independent axes)
  [APPX] density_pct      dis=0.550 adv=0.517 diff=+0.033  rb=-0.052  p=3.272e-01
  [APPX] A_LowDensity     dis=0.450 adv=0.483 diff=-0.033  rb=+0.052  p=3.272e-01
  [APPX] A_NonFrag        dis=0.457 adv=0.529 diff=-0.072  rb=+0.197  p=1.949e-04 *
T2 subsampling control is DROPPED (R-17).
Reason: fixing the smaller group understates variability; the two-sided
bootstrap in 07_robustness.ipynb is used for the Stability gap CI.
T3 (RQ3) — stability gap vs the approval-rate gap (BOUNDED):
  Groups were defined by low approval rate AND demographic cells.
  Bounded claim: a stability difference exists tha